# Satellite Propagation Models
---

This notebook walks through the satellite and 3GPP propagation models in `xant.propagation.satcom`, first as
stand-alone formulas and then inside a spatial link.

* Compare what 3GPP TR 38.811 adds on top of free space: line-of-sight probability, shadow fading and clutter loss by scenario and elevation angle
* Sweep frequency through the ITU-R slant-path models (P.676 gases, P.840 cloud, P.618 rain and scintillation) to see where the atmosphere starts to matter
* Look at the land-mobile and clutter models: ITU-R P.681 roadside tree shadowing and ITU-R P.2108 earth-space clutter
* Run a LEO downlink with `calculate_spatial_link` under each propagator and compare the received power against elevation
* Map the received power over a region with `transmit_power_density`

Everything here runs offline: coordinate systems are created with `hagl=False`, so no terrain is downloaded. The
atmospheric models need the optional `itur` package (`pip install xant[itur]`).

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from hics import HCS

from xant import ureg
from xant.antenna import common
from xant.propagation import rflink
from xant.propagation import satcom
from xant.propagation.itm.p2108 import p2108_3

# from xant import XANTLogger
# XANTLogger.unmute()
# XANTLogger.level = "DEBUG"

# 3GPP TR 38.811
---

TR 38.811 section 6.6 builds the path loss of a satellite or HAPS link from four terms

$$
\begin{equation}
PL = PL_b + PL_g + PL_s + PL_e
\end{equation}
$$

where $PL_b$ is the basic path loss, $PL_g$ the attenuation by atmospheric gases, $PL_s$ scintillation and $PL_e$
building entry loss. The basic path loss is free space plus a log-normal shadow fading term and, when the path is
blocked, a clutter loss

$$
\begin{equation}
PL_b = FSPL(d, f_c) + SF + CL(\alpha, f_c), \qquad SF \sim N(0, \sigma_{SF}^2)
\end{equation}
$$

$\sigma_{SF}$ and $CL$ are tabulated against the elevation angle $\alpha$ for dense urban, urban and
suburban / rural scenarios, in S band and Ka band.

## Line-of-sight probability

Whether the clutter loss applies depends on whether the terminal sees the satellite. Table 6.6.1-1 gives that
probability, and it is the main reason low elevation angles are expensive in built-up areas.

In [ ]:
elevation = np.arange(10, 91, 10)

fig, ax = plt.subplots()
for scenario in ("dense_urban", "urban", "suburban"):
    ax.plot(elevation, 100 * satcom.ntn_los_probability(elevation, scenario), "o-", label=scenario)
ax.set_xlabel("Elevation [deg]")
ax.set_ylabel("LOS probability [%]")
ax.set_title("TR 38.811 Table 6.6.1-1")
ax.legend()
ax.grid()

## Shadow fading and clutter loss

`ntn_shadow_clutter` returns the table values directly: $\sigma_{SF}$ for LOS, $\sigma_{SF}$ for NLOS, and the
clutter loss. The S-band columns are used below `band_split_ghz` (6 GHz by default) and the Ka-band columns above.

In [ ]:
for f_ghz in (2.0, 20.0):
    sig_los, sig_nlos, clutter = satcom.ntn_shadow_clutter(f_ghz, 30.0, "urban")
    print(
        f"{f_ghz:4.0f} GHz, urban, 30 deg: sigma_LOS = {sig_los} dB, sigma_NLOS = {sig_nlos} dB, CL = {clutter} dB"
    )

## Excess loss as a location percentile

A link budget needs one number, not a distribution. `ntn_excess_loss` returns $SF + CL$ *not exceeded* at a given
percentage of locations, so a larger percentage is a more conservative loss. The `los` argument selects the state:

* `True` - the terminal has line of sight, only shadow fading applies
* `False` - the path is blocked, clutter loss plus the wider NLOS shadow fading
* `"mixture"` - the state is random with the probability of Table 6.6.1-1 and the percentile is taken over the
  combined distribution. This is the one to use for a coverage percentage when the state is unknown.

The mixture is bimodal, which is why its median jumps from the LOS curve to the NLOS curve where the LOS
probability crosses 50%.

In [ ]:
elevation = np.linspace(10, 90, 81)

fig, axs = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, scenario in zip(axs, ("urban", "suburban")):
    for los, ls in ((True, ":"), (False, "--"), ("mixture", "-")):
        for location, color in ((50, "C0"), (90, "C1")):
            loss = satcom.ntn_excess_loss(2.0, elevation, scenario, los=los, location=location)
            ax.plot(elevation, loss, ls=ls, color=color, label=f"los={los}, {location}%")
    ax.set_title(f"{scenario}, S band")
    ax.set_xlabel("Elevation [deg]")
    ax.grid()
axs[0].set_ylabel("Shadow fading + clutter [dB]")
axs[1].legend(fontsize=8)

# ITU-R Slant-Path Atmosphere
---

Above roughly 10 GHz the atmosphere becomes the term that sizes the link. `atmosphere_slant_path` wraps the `itur`
package and returns the attenuation exceeded for `p` percent of an average year at a given site, split into

* gases - ITU-R P.676
* cloud - ITU-R P.840
* rain - ITU-R P.618 with the P.837 rain-rate and P.839 rain-height maps
* scintillation - ITU-R P.618
* total - combined as in P.618 section 2.5 (not a plain sum)

## Frequency sweep

Boulder, 30 degrees elevation, 1.2 m dish, attenuation exceeded for 1% and 0.1% of the year.

In [ ]:
site_lat, site_lon = 40.015, -105.27
freqs_ghz = np.array([1, 2, 4, 6, 8, 10, 12, 14, 18, 20, 22.2, 24, 28, 30, 35, 40, 45, 50.0])

fig, axs = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, p in zip(axs, (1.0, 0.1)):
    sweep = [
        satcom.atmosphere_slant_path(site_lat, site_lon, f, 30.0, p=p, antenna_diameter_m=1.2)
        for f in freqs_ghz
    ]
    for key in ("gas", "cloud", "rain", "scintillation", "total"):
        ax.semilogy(
            freqs_ghz,
            [float(s[key]) for s in sweep],
            "k-" if key == "total" else "-",
            lw=2 if key == "total" else 1,
            label=key,
        )
    ax.set_title(f"Exceeded {p}% of the year")
    ax.set_xlabel("Frequency [GHz]")
    ax.grid(which="both", alpha=0.4)
axs[0].set_ylabel("Attenuation [dB]")
axs[0].set_ylim(1e-2, 1e2)
axs[1].legend()

The 22.2 GHz water-vapour line shows up in the gas curve, and rain takes over from gases as the dominant term
somewhere between 10 and 20 GHz depending on the availability target.

## Availability

The same call swept over `p` gives the fade margin needed for a target availability.

In [ ]:
p_values = np.array([5, 3, 2, 1, 0.5, 0.3, 0.2, 0.1, 0.05, 0.03, 0.01])

fig, ax = plt.subplots()
for f_ghz in (12.0, 20.0, 30.0):
    total = [
        float(
            satcom.atmosphere_slant_path(
                site_lat, site_lon, f_ghz, 30.0, p=p, antenna_diameter_m=1.2
            )["total"]
        )
        for p in p_values
    ]
    ax.loglog(p_values, total, "o-", label=f"{f_ghz:.0f} GHz")
ax.set_xlabel("Percentage of the year exceeded [%]")
ax.set_ylabel("Total atmospheric attenuation [dB]")
ax.set_title("Boulder, 30 deg elevation")
ax.invert_xaxis()
ax.legend()
ax.grid(which="both", alpha=0.4)

TR 38.811 itself uses a simpler gas term: the zenith attenuation of the P.676 standard atmosphere divided by
$\sin\alpha$. That is `atmosphere_gas_standard`, and it is what `ntn_tr38811(atmosphere=True)` adds.

In [ ]:
for f_ghz in (2.0, 12.0, 20.0, 30.0):
    zenith, low = satcom.atmosphere_gas_standard(f_ghz, [90.0, 10.0])
    print(f"{f_ghz:4.0f} GHz: {zenith:.3f} dB at zenith, {low:.2f} dB at 10 deg")

# Land-Mobile and Clutter Models
---

## ITU-R P.681 roadside tree shadowing

For a vehicle terminal on a tree-lined road, P.681 section 4.1.1 gives the fade *exceeded* for `p` percent of the
distance travelled. Note the opposite sense to the percentiles above: a small `p` is a deep, rare fade.

The model covers 0.8 to 20 GHz and 7 to 60 degrees elevation. Above 60 degrees the Recommendation only gives data
for 1.6 to 2.6 GHz, so the function returns NaN elsewhere instead of extrapolating.

In [ ]:
p = np.array([1, 2, 5, 10, 20, 30, 50, 80])

fig, axs = plt.subplots(1, 2, figsize=(11, 4))
for elev in (20, 30, 45, 60):
    axs[0].semilogx(p, satcom.p681_roadside_tree_shadowing(1.5, elev, p), "o-", label=f"{elev} deg")
axs[0].set_title("1.5 GHz")
axs[0].set_xlabel("Percentage of distance fade is exceeded [%]")
axs[0].set_ylabel("Fade [dB]")
axs[0].legend(title="Elevation")
axs[0].grid(which="both", alpha=0.4)

f_ghz = np.linspace(0.8, 20, 100)
for pi in (1, 10, 20):
    axs[1].plot(f_ghz, satcom.p681_roadside_tree_shadowing(f_ghz, 45.0, pi), label=f"p = {pi}%")
axs[1].set_title("45 deg elevation")
axs[1].set_xlabel("Frequency [GHz]")
axs[1].legend()
axs[1].grid()

## ITU-R P.2108 earth-space clutter

P.2108 section 3.3 is the ITU-R counterpart to the TR 38.811 clutter table: a statistical clutter loss for a
terminal inside urban or suburban clutter, 10 to 100 GHz, as a function of elevation. It returns the loss not
exceeded for a fraction `p` of locations.

In [ ]:
elevation = np.linspace(0, 90, 91) * ureg.degree

fig, ax = plt.subplots()
for p in (0.1, 0.5, 0.9):
    ax.plot(
        elevation.magnitude,
        p2108_3(30 * ureg.GHz, elevation, p).magnitude,
        label=f"{100 * p:.0f}% of locations",
    )
ax.set_xlabel("Elevation [deg]")
ax.set_ylabel("Clutter loss [dB]")
ax.set_title("ITU-R P.2108 section 3.3, 30 GHz")
ax.legend()
ax.grid()

# LEO Downlink
---

Now the same models inside a link. Each one is a propagator with the same signature as `fspl`, so it is selected
by name in `calculate_spatial_link`. The elevation angle is worked out from the two coordinate systems.

## Setup coordinate systems

One satellite at 600 km above Boulder, and a line of ground terminals running north from the sub-satellite point
so the link sweeps from zenith down to the horizon.

In [ ]:
sat_cs = HCS.from_crs((40.015 * ureg.degree, -105.27 * ureg.degree, 600 * ureg.km), hagl=False)

npts = 60
lat = xr.DataArray(np.linspace(40.015, 60.0, npts) * ureg.degree, dims="terminal")
lon = xr.DataArray(np.full(npts, -105.27) * ureg.degree, dims="terminal")
alt = xr.DataArray(np.full(npts, 1600.0) * ureg.m, dims="terminal")
ground_cs = HCS.from_crs([lat, lon, alt], hagl=False)

geometry = satcom.link_geometry(sat_cs, ground_cs)
elevation = geometry["elevation"]
elevation.values[[0, npts // 2, -1]]

## Setup RF parameters and antennas

Two downlink frequencies, S band and Ka band. Isotropic antennas at both ends keep the comparison about
propagation only; swap in any `Antenna` or `AntennaArray.total` to include the patterns.

In [ ]:
freq = np.array([2.0, 20.0]) * ureg.GHz
tx_power = 20 * ureg.watt

sat = common.Isotropic(frequency=freq, hcs=sat_cs)
terminal = common.Isotropic(frequency=freq, hcs=ground_cs)

## Compute the link under each model

In [ ]:
cases = {
    "Free space": dict(propagation="fspl"),
    "TR 38.811 suburban, 50%": dict(
        propagation="ntn_tr38811", scenario="suburban", location=50, atmosphere=True
    ),
    "TR 38.811 urban, 50%": dict(
        propagation="ntn_tr38811", scenario="urban", location=50, atmosphere=True
    ),
    "TR 38.811 urban, 90%": dict(
        propagation="ntn_tr38811", scenario="urban", location=90, atmosphere=True
    ),
    "ITU-R slant path, 1%": dict(
        propagation="earth_space_itur", p=1.0, antenna_diameter=0.6 * ureg.m
    ),
    "ITU-R slant path, 0.1%": dict(
        propagation="earth_space_itur", p=0.1, antenna_diameter=0.6 * ureg.m
    ),
    "P.681 trees, 10%": dict(propagation="lms_p681", percent_exceeded=10),
}

results = {}
for name, kwargs in cases.items():
    res, txcs, rxcs = rflink.calculate_spatial_link(sat, tx_power, terminal, **kwargs)
    results[name] = res
results["TR 38.811 urban, 90%"]

The individual terms ride along as coordinates of `prop_loss` (`fspl`, `shadow_clutter`, `gas`, `elevation`,
`los_probability`, ...), so the budget can be broken down after the fact.

In [ ]:
def to_units(da, units):
    # Convert a Quantity DataArray to plain floats in the requested units
    out = da.copy()
    out.data = out.data.to(units).magnitude
    return out


fig, axs = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, f in zip(axs, freq.to("Hz").magnitude):
    for name, res in results.items():
        rx = to_units(res.rx_power, "dBm").sel(frequency=f)
        ax.plot(elevation, rx, label=name)
    ax.set_title(f"{f / 1e9:.0f} GHz")
    ax.set_xlabel("Elevation [deg]")
    ax.set_xlim(0, 90)
    ax.grid()
axs[0].set_ylabel("Received power [dBm]")
axs[1].legend(fontsize=8)

A few things to read off:

* At S band the atmosphere is negligible and the scenario decides everything. Urban at 50% steps down by the
  clutter loss where the LOS probability drops under one half.
* At Ka band the ITU-R curves pull away from free space at low elevation, where the path through the rain and gas
  layers gets long.
* The P.681 curve stops outside 7 to 60 degrees at 20 GHz because the model has no data there.

## Loss breakdown

Pull the terms out of one result to see the budget at a single elevation.

In [ ]:
res = results["TR 38.811 urban, 90%"]
idx = int(np.abs(elevation.values - 30).argmin())
loss = res.prop_loss.isel(terminal=idx)

print(
    f"Elevation {float(loss.elevation):.1f} deg, LOS probability {100 * float(loss.los_probability):.0f}%"
)
for f in freq.to("Hz").magnitude:
    row = loss.sel(frequency=f)
    print(
        f"{f / 1e9:4.0f} GHz: FSPL {float(row.fspl):6.1f} dB"
        f" + shadow/clutter {float(row.shadow_clutter):5.1f} dB"
        f" + gas {float(row.gas):4.2f} dB"
        f" = {float(row.data.to('dB').magnitude):6.1f} dB",
    )

# Coverage Map
---

`transmit_power_density` projects the transmit power onto a latitude / longitude grid. Model options go in
`propagation_kwargs`. Here the S-band downlink is mapped for free space and for a blocked dense-urban terminal at
90% of locations (note the two colour scales are about 40 dB apart); the rings in the second map are the 10 degree elevation steps of the TR 38.811 tables (pass
`interpolate=True` to smooth them).

In [ ]:
sat_s = common.Isotropic(frequency=freq[:1], hcs=sat_cs)
lats = np.linspace(32, 48, 81) * ureg.degree
lons = np.linspace(-116, -94, 111) * ureg.degree
hs = [1600] * ureg.m

maps = {}
for name, pk in {
    "Free space": dict(propagation="fspl"),
    "TR 38.811 dense urban NLOS, 90%": dict(
        propagation="ntn_tr38811",
        propagation_kwargs=dict(scenario="dense_urban", los=False, location=90),
    ),
}.items():
    pd = rflink.transmit_power_density(sat_s, tx_power, lats, lons, hs, mask_los=False, **pk)
    maps[name] = to_units(pd.sel(polarization="apolar").squeeze(), "dBm")

fig, axs = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, (name, m) in zip(axs, maps.items()):
    # Each map gets its own 30 dB colour span below its peak
    vmax = np.ceil(float(m.max()))
    m.transpose("lat", "lon").plot(
        ax=ax, vmin=vmax - 30, vmax=vmax, cbar_kwargs=dict(label="Received power [dBm]")
    )
    ax.plot(-105.27, 40.015, "k+", ms=10)
    ax.set_title(name)
    ax.set_xlabel("Longitude [deg]")
    ax.set_ylabel("Latitude [deg]")

# 3GPP TR 38.901 (terrestrial)
---

For the ground segment or a terrestrial comparison, the TR 38.901 scenarios cover 0.5 to 100 GHz. Free-space loss
is already inside these formulas.

In [ ]:
d2d = np.logspace(1, np.log10(5000), 200)

fig, axs = plt.subplots(1, 2, figsize=(11, 4))
for scenario in ("rma", "uma", "umi"):
    (line,) = axs[0].semilogx(
        d2d, satcom.tr38901_path_loss(scenario, d2d, 3.5, los=True), label=f"{scenario} LOS"
    )
    axs[0].semilogx(
        d2d,
        satcom.tr38901_path_loss(scenario, d2d, 3.5, los=False),
        "--",
        color=line.get_color(),
        label=f"{scenario} NLOS",
    )
    axs[1].semilogx(
        d2d,
        100 * satcom.tr38901_los_probability(scenario, d2d),
        color=line.get_color(),
        label=scenario,
    )
axs[0].set_title("Path loss, 3.5 GHz")
axs[0].set_xlabel("2D distance [m]")
axs[0].set_ylabel("Path loss [dB]")
axs[0].legend(fontsize=8)
axs[0].grid(which="both", alpha=0.4)
axs[1].set_title("LOS probability")
axs[1].set_xlabel("2D distance [m]")
axs[1].set_ylabel("LOS probability [%]")
axs[1].legend()
axs[1].grid(which="both", alpha=0.4)